# 알고리즘 기말 프로젝트 — Score Function Design

- **제출일**: `<YYYY.MM.DD HH:MM>`
- **파일명**: `이름_학번_EDC_like_agrochemical.ipynb`  (예: 신현길_12312312_EDC_like_agrochemical.ipynb)

## 학번 / 이름

- **학번**: `          `
- **이름**: `          `
- **score에 대한 간략한 설명**: Agrochemical reference set을 기준으로 physicochemical property, Murcko scaffold, fingerprint similarity, SMARTS fragment enrichment를 결합하여 ZINC 후보의 EDC-like agrochemical-likeness를 계산하는 explainable score.




## 설명 가능한 EDC-like Agrochemical Scoring System

- 데이터: `PubChem_Agrochemical.csv`, `zinc_db/*.txt`
- 비교 데이터: `PubChem_Drug.csv`
- 목표: 단순 black-box classifier가 아니라 **physicochemical property + scaffold + similarity + fragment** 근거를 조합한 설명 가능한 scoring system 만들기
- 최종 점수: weighted geometric mean

### 핵심 질문

이 분자는 왜 EDC-like 후보인가?

최종 답은 단순히 `score = 0.83`에서 끝나면 안 된다. 아래 화학적 근거를 함께 설명해야 한다.

1. Property: 분자의 물성이 agrochemical reference distribution 안에 들어오는가?
2. Scaffold: agrochemical에서 enriched된 Murcko scaffold를 포함하는가?
3. Similarity: Tanimoto fingerprint 기준으로 알려진 agrochemical과 유사한가?
4. Fragment: agrochemical 또는 EDC-like chemical alert와 관련된 SMARTS fragment를 포함하는가?


## 채점 기준 (100점)

| 영역 | 배점 | 만점 기준 |
|---|---|---|
| **(1) Negative 데이터 준비** | 20 | "구조 유사도"를 통해 negative 집합을 찾아낸 기준? (유사도 측정 방법 & Structural similarity 기준 설정) |
| **(2) Score 함수 설계** | 20 | **(a) 분자 속성 범위 (전체 데이터에서 property 범위 계산 방식)** + **(b) alert 구조 패턴(scaffold 및 부분 구조의 smarts 패턴)** 두 요소 모두 포함. score에 대한 설명은 markdown에 기재. |
| **(3) Score 평가 — goodness** | 20 | positive vs negative 점수 분포 비교 (히스토그램/ROC/AUC 등 score 성능의 근거가 되는 시각화 자료 제시) |
| **(4) 설명** | 10 | 각 알고리즘을 mermaid를 이용해서 표현하고 설명글 추가 (markdown 및 주석으로 표기) |


### 가산점 (선택)

| 가산 | 점수 | 조건 |
|---|---|---|
| **(A) 다른 화학 제품군 score** | +10 | pesticide 외 1종 이상(cosmetic / food additive / fragrance / surfactant / dye 등)의 PubChem classification 데이터로 별도 score 함수 설계 + 평가 |
| **(B) Score 기반 구조 생성** | +10 | 본인 score 를 reward 로 사용해 score가 개선된 새로운 구조 생성. |
| **(C) 계산 자원과 계산 시간** | +10 | mpi를 이용해서 대량의 자원으로 계산 시간을 대폭 줄이거나, local 환경에서 합리적으로 계산이 진행될 수 있도록 문제를 효율적으로 압축시킨 방법 적용 (mpi script와 계산 결과에 대한 log 필요) |

### 제출 결과물 (결과를 재현하기 위해 필요한 파일들)
1. ipynb (mpi를 사용했다면, mpi4py script)
2. data files (pesticide, cosmetics, food additives, drug, ..., format: csv)
3. negative list file (format: csv)
4. score 평가 시각화 자료 (mpi에서 실행해서 얻은 plot은 notebook markdown에 삽입)

---
# Task 1. Negative 데이터 준비 (25점)

**문제**: 양성(positive) 분자와 "구조적으로 다른" 분자 집합을 어떻게 만들 것인가?

Score 함수의 평가는 **양성과 음성을 얼마나 잘 구분하는가** 로 측정합니다. 그러려면 먼저 음성 집합을 정의해야 합니다.

**📝 본인 선택과 이유 (직접 작성):**

- 선택한 기준: positive set은 `PubChem_Agrochemical.csv`로 정의하고, comparison/negative set은 `PubChem_Drug.csv`를 사용한다. ZINC 후보는 screening 대상이며, 평가 과정에서는 agrochemical reference와의 Tanimoto similarity가 낮고 agrochemical-enriched scaffold/fragment가 부족한 분자를 실질적 negative-like 후보로 본다.
- 이유: 이 프로젝트의 목적은 단순한 독성 분류기가 아니라 “왜 EDC-like agrochemical 후보인지” 설명 가능한 score를 만드는 것이다. 따라서 positive는 실제 agrochemical annotation을 가진 분자로 잡고, negative/comparison은 생리활성은 있을 수 있지만 agrochemical 용도와 구조공간이 다른 drug dataset으로 잡았다. Drug set을 비교군으로 쓰면 단순히 “bioactive molecule”을 찾는 것이 아니라 agrochemical-like structural/property signature를 구분하는지 평가할 수 있다.

**구조 유사도 기준 설정:**

- Fingerprint는 Morgan(radius=2, 2048 bits), RDKit fingerprint, MACCS keys를 비교한다.
- 각 후보 분자에 대해 agrochemical reference set과의 최대 Tanimoto similarity를 계산한다.
- Similarity distribution에서 drug와 agrochemical의 분포가 갈라지는 구간을 확인하고, ZINC 후보 중 similarity가 높으면서도 property/scaffold/fragment score가 함께 높은 분자를 EDC-like 후보로 우선순위화한다.
- 단일 threshold만 사용하지 않고, similarity score를 final weighted geometric mean의 한 구성 요소로 사용한다. 이렇게 하면 유사도는 높지만 물성이나 fragment 근거가 약한 분자가 과도하게 선택되는 문제를 줄일 수 있다.

**Task 1 결과 해석 방향:**

- 좋은 negative/comparison set이라면 drug score distribution은 agrochemical distribution보다 낮거나 넓게 분산되어야 한다.
- Tanimoto similarity histogram에서 agrochemical self-similarity는 높은 쪽에, drug-to-agrochemical similarity는 상대적으로 낮은 쪽에 위치할 것으로 예상한다.
- 두 분포가 완전히 분리되지 않는 것은 정상이다. 일부 drug는 agrochemical과 scaffold 또는 fragment를 공유할 수 있기 때문이다. 따라서 final score는 similarity 하나가 아니라 property, scaffold, fragment evidence를 함께 요구하도록 설계한다.


In [ ]:
from pathlib import Path
import math
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

from rdkit import Chem, DataStructs
from rdkit.Chem import Descriptors, Crippen, Lipinski, rdMolDescriptors, MACCSkeys
from rdkit.Chem.Scaffolds import MurckoScaffold
from rdkit.Chem import Draw

try:
    from sklearn.neighbors import KernelDensity
    from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, average_precision_score
    SKLEARN_OK = True
except Exception:
    SKLEARN_OK = False

BASE = Path(r"C:\Users\DS\AL")
PROJECT = BASE / "project"
AGRO_PATH = BASE / "PubChem_Agrochemical.csv"
DRUG_PATH = BASE / "PubChem_Drug.csv"
ZINC_DIR = BASE / "zinc_db"

RANDOM_SEED = 42
MAX_AGRO = 5000
MAX_DRUG = 5000
MAX_ZINC_FILES = 20
MAX_ZINC = 10000

np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)


## 1. Data Loading

PubChem CSV 파일은 데이터셋마다 컬럼명이 다를 수 있다. 아래 helper 함수는 SMILES, name, CID 컬럼을 자동으로 찾아서 사용한다.

ZINC 데이터는 수업에서 사용한 `zinc_db/*.txt` tab-separated 형식을 그대로 읽는다.


In [ ]:
def pick_col(df, candidates):
    lower = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    return None

def load_pubchem_csv(path, label, max_rows=None):
    df = pd.read_csv(path)
    smi_col = pick_col(df, ["SMILES", "smiles", "CanonicalSMILES", "canonical_smiles"])
    name_col = pick_col(df, ["Name", "cmpdname", "IUPAC_Name", "iupacname"])
    cid_col = pick_col(df, ["Compound_CID", "cid", "CID"])
    if smi_col is None:
        raise ValueError(f"SMILES column not found in {path}")

    out = pd.DataFrame({
        "source": label,
        "smiles": df[smi_col].astype(str),
        "name": df[name_col].astype(str) if name_col else "",
        "cid": df[cid_col].astype(str) if cid_col else "",
    })
    out = out[out["smiles"].notna() & (out["smiles"] != "nan")].drop_duplicates("smiles")
    if max_rows and len(out) > max_rows:
        out = out.sample(max_rows, random_state=RANDOM_SEED)
    return out.reset_index(drop=True)

def load_zinc_dir(zinc_dir, max_files=20, max_rows=10000):
    frames = []
    files = sorted(Path(zinc_dir).glob("*.txt"))[:max_files]
    for file in files:
        part = pd.read_csv(file, sep="\t")
        if "smiles" not in part.columns:
            continue
        name_col = "zinc_id" if "zinc_id" in part.columns else None
        frames.append(pd.DataFrame({
            "source": "zinc",
            "smiles": part["smiles"].astype(str),
            "name": part[name_col].astype(str) if name_col else file.stem,
            "cid": "",
        }))
    zinc = pd.concat(frames, ignore_index=True).drop_duplicates("smiles")
    if len(zinc) > max_rows:
        zinc = zinc.sample(max_rows, random_state=RANDOM_SEED)
    return zinc.reset_index(drop=True)

agro_raw = load_pubchem_csv(AGRO_PATH, "agrochemical", MAX_AGRO)
drug_raw = load_pubchem_csv(DRUG_PATH, "drug", MAX_DRUG)
zinc_raw = load_zinc_dir(ZINC_DIR, MAX_ZINC_FILES, MAX_ZINC)

print("agro:", agro_raw.shape)
print("drug:", drug_raw.shape)
print("zinc:", zinc_raw.shape)
display(pd.concat([agro_raw.head(3), drug_raw.head(3), zinc_raw.head(3)]))


In [ ]:
def mol_from_smiles(smi):
    try:
        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return None
        return mol
    except Exception:
        return None

def canonical_smiles(mol):
    return Chem.MolToSmiles(mol, isomericSmiles=True) if mol is not None else None

def clean_molecules(df):
    df = df.copy()
    df["mol"] = df["smiles"].map(mol_from_smiles)
    df = df[df["mol"].notna()].copy()
    df["canonical_smiles"] = df["mol"].map(canonical_smiles)
    df = df.drop_duplicates("canonical_smiles").reset_index(drop=True)
    return df

agro = clean_molecules(agro_raw)
drug = clean_molecules(drug_raw)
zinc = clean_molecules(zinc_raw)

print("valid agro/drug/zinc:", len(agro), len(drug), len(zinc))


## 2. Descriptor Calculation and Distribution

이 단계는 수업에서 다룬 descriptor 기반 scoring과 연결된다. PubChem 원본 컬럼 대신 RDKit으로 descriptor를 다시 계산하여 agrochemical, drug, ZINC 데이터에 동일한 기준을 적용한다.


In [ ]:
def calc_descriptors(mol):
    return {
        "MW": Descriptors.MolWt(mol),
        "LogP": Crippen.MolLogP(mol),
        "TPSA": rdMolDescriptors.CalcTPSA(mol),
        "HBD": Lipinski.NumHDonors(mol),
        "HBA": Lipinski.NumHAcceptors(mol),
        "RotB": Lipinski.NumRotatableBonds(mol),
        "HeavyAtoms": mol.GetNumHeavyAtoms(),
        "AromaticRings": rdMolDescriptors.CalcNumAromaticRings(mol),
    }

def add_descriptors(df):
    desc = pd.DataFrame([calc_descriptors(m) for m in df["mol"]])
    return pd.concat([df.reset_index(drop=True), desc], axis=1)

agro = add_descriptors(agro)
drug = add_descriptors(drug)
zinc = add_descriptors(zinc)

desc_cols = ["MW", "LogP", "TPSA", "HBD", "HBA", "RotB", "HeavyAtoms", "AromaticRings"]
all_desc = pd.concat([agro, drug, zinc], ignore_index=True)
display(all_desc.groupby("source")[desc_cols].describe().round(2))


In [ ]:
plot_df = pd.concat([
    agro.assign(group="Agrochemical"),
    drug.assign(group="Drug comparison"),
    zinc.assign(group="ZINC candidate"),
], ignore_index=True)

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.ravel()
for ax, col in zip(axes, desc_cols):
    if sns:
        sns.kdeplot(data=plot_df, x=col, hue="group", common_norm=False, ax=ax, warn_singular=False)
    else:
        for group, part in plot_df.groupby("group"):
            ax.hist(part[col].dropna(), bins=40, alpha=0.35, density=True, label=group)
        ax.legend(fontsize=8)
    ax.set_title(col)
plt.tight_layout()
plt.show()


### 해석: descriptor distribution

- Agrochemical은 일반적으로 drug보다 분자량과 TPSA 범위가 더 넓게 나타날 수 있고, halogenated aromatic 또는 lipophilic fragment 때문에 LogP가 높게 나타나는 후보가 존재한다. 반면 drug set은 oral drug-likeness 제약 때문에 HBD/HBA, MW, TPSA가 상대적으로 규칙적인 범위에 모이는 경향이 있다.
- ZINC 후보 중 EDC-like agrochemical 후보로 볼 수 있는 영역은 MW, LogP, TPSA, HBD/HBA, RotB가 agrochemical reference distribution의 중심 또는 high-density 영역에 들어오는 부분이다.
- 본 score에서 중요한 descriptor는 `LogP`, `TPSA`, `MW`, `HBD/HBA`, `RotB`이다. EDC-like 가능성은 특정 receptor 또는 membrane interaction과 관련될 수 있으므로 hydrophobicity(LogP), polar surface area(TPSA), hydrogen bonding pattern이 중요하다.
- Property score는 각 descriptor의 KDE/Gaussian desirability를 계산한 뒤 geometric mean으로 결합한다. 따라서 하나의 물성만 좋아도 높은 점수가 되는 것이 아니라 여러 물성이 agrochemical-like 범위에 동시에 들어와야 높은 점수가 된다.


---
# Task 2. Score 함수 설계 (35점)

**문제**: "positive-likeness" 점수를 계산해주는 함수 개발.

**Scoring 방식**
1. **(a) 분자 속성 범위** — MW, logP, HBA, HBD, TPSA, rotatable bonds 등 descriptor를 사용하여 agrochemical-like property range를 정의한다.
2. **(b) SMARTS 패턴** — positive set에서 자주 나타나는 작용기/하부구조를 포함하고, drug comparison set에서도 흔한 일반 패턴은 enrichment로 보정한다.
3. **(c) Scaffold와 similarity** — Murcko scaffold와 fingerprint Tanimoto similarity를 추가하여 단순 property filter보다 구조적 설명력이 높은 score를 만든다.

두 점수만 단순 합산하지 않고, 본 프로젝트에서는 **property + scaffold + similarity + fragment** 네 요소를 weighted geometric mean으로 결합한다.

## 📝 Score 함수 설계 근거 및 답안

### 1. Property score

Agrochemical positive set의 descriptor distribution을 reference로 사용한다. 각 후보 분자의 MW, LogP, TPSA, HBD, HBA, RotB, AromaticRings 값이 agrochemical reference distribution에서 얼마나 자연스러운지 Gaussian 또는 KDE desirability로 계산한다.

- Gaussian desirability: reference 평균과 표준편차를 이용해 z-score가 작을수록 높은 점수
- KDE desirability: reference distribution의 밀도가 높은 위치일수록 높은 점수
- Descriptor별 점수는 geometric mean으로 결합

이 방식은 특정 descriptor 하나만 맞는 분자가 아니라 여러 물성이 동시에 agrochemical-like인 분자를 선호한다.

### 2. Scaffold score

Murcko scaffold를 추출하고 agrochemical set과 drug comparison set에서의 빈도를 비교한다.

- Agrochemical에서 자주 등장하는 scaffold: score 증가
- Drug set에서도 흔한 scaffold: specificity가 낮으므로 penalty
- Scaffold가 없거나 reference에 거의 없는 경우: 낮은 baseline score

따라서 scaffold score는 “이 분자가 agrochemical에서 반복적으로 관찰되는 core structure를 가지는가?”에 대한 설명 가능한 근거가 된다.

### 3. Similarity score

Morgan, RDKit, MACCS fingerprint를 각각 계산하고 agrochemical reference set과의 maximum Tanimoto similarity를 구한다.

- Morgan: local circular environment 반영
- RDKit: path-based structural pattern 반영
- MACCS: 해석 가능한 structural key 반영

최종 similarity score는 세 fingerprint similarity의 평균으로 계산한다. 하나의 fingerprint에만 의존하지 않기 때문에 특정 표현 방식의 편향을 줄일 수 있다.

### 4. Fragment score

SMARTS library를 사용하여 phenol, halogenated aromatic, nitro aromatic, amide, urea, carbamate, triazine, phosphate, sulfone/sulfonamide, nitrile, pyridine, azole 등의 fragment hit를 계산한다.

Fragment score는 단순 hit 개수가 아니라 agrochemical vs drug enrichment를 반영한다. Agrochemical에서 더 자주 나타나는 fragment는 높은 contribution을 주고, drug에도 흔한 fragment는 상대적으로 낮은 contribution을 갖는다.

### 5. 최종 결합 방식

최종 점수는 weighted geometric mean으로 계산한다.

```text
Final score = exp(sum(w_i * log(score_i + eps)) / sum(w_i))
```

기본 가중치는 다음과 같다.

- property_score: 0.30
- scaffold_score: 0.20
- similarity_score: 0.30
- fragment_score: 0.20

Weighted geometric mean을 선택한 이유는 하나의 subscore만 높은 분자가 최종 상위 후보가 되는 것을 막기 위해서다. EDC-like 후보는 물성, scaffold, similarity, fragment evidence가 함께 지지될 때 높은 점수를 받아야 한다.

### 6. 선택한 SMARTS 패턴과 이유

- **phenol**: endocrine receptor interaction과 관련된 phenolic motif 가능성을 확인하기 위해 포함
- **halogenated_aromatic**: 농약/환경화학물질에서 자주 관찰되는 hydrophobic aromatic motif
- **nitro_aromatic**: 일부 pesticide/toxicophore 관련 구조 확인
- **urea/carbamate/triazine/phosphate**: pesticide class에서 자주 등장하는 functional motif
- **amide, nitrile, pyridine, azole**: agrochemical과 bioactive small molecule에서 중요한 heteroatom-containing motif

이 fragment들은 “왜 EDC-like 후보로 의심되는가?”를 substructure 수준에서 설명하기 위한 alert 역할을 한다. 단, fragment hit만으로 EDC 활성을 확정하지 않고, property/scaffold/similarity evidence와 함께 해석한다.


## 3. Property Score: Gaussian or KDE Desirability

Property score는 각 descriptor 값이 agrochemical reference distribution에서 얼마나 자연스러운지 측정한다.

- Gaussian: 평균과 표준편차를 이용한 z-score 기반 desirability
- KDE: empirical reference distribution의 밀도를 이용한 desirability

최종 property score는 descriptor별 desirability의 geometric mean으로 계산한다.


In [ ]:
EPS = 1e-9

def gaussian_desirability(values, ref_values):
    ref = pd.Series(ref_values).dropna().astype(float)
    mu = ref.mean()
    sd = ref.std()
    if not np.isfinite(sd) or sd < EPS:
        sd = 1.0
    z = (pd.Series(values).astype(float) - mu) / sd
    return np.exp(-0.5 * z * z).clip(EPS, 1.0)

def kde_desirability(values, ref_values):
    if not SKLEARN_OK:
        return gaussian_desirability(values, ref_values)
    ref = pd.Series(ref_values).dropna().astype(float).values.reshape(-1, 1)
    val = pd.Series(values).astype(float).values.reshape(-1, 1)
    if len(ref) < 30 or np.std(ref) < EPS:
        return gaussian_desirability(values, ref_values)
    bandwidth = max(0.05, 0.25 * np.std(ref))
    kde = KernelDensity(kernel="gaussian", bandwidth=bandwidth).fit(ref)
    log_dens = kde.score_samples(val)
    ref_log_dens = kde.score_samples(ref)
    scaled = np.exp(log_dens - np.percentile(ref_log_dens, 95))
    return pd.Series(np.clip(scaled, EPS, 1.0))

def weighted_geometric_mean(score_frame, weights):
    cols = list(weights.keys())
    w = np.array([weights[c] for c in cols], dtype=float)
    w = w / w.sum()
    x = score_frame[cols].clip(EPS, 1.0).values
    return np.exp(np.dot(np.log(x), w))

property_weights = {
    "MW": 1.0,
    "LogP": 1.2,
    "TPSA": 1.0,
    "HBD": 0.8,
    "HBA": 0.8,
    "RotB": 0.8,
    "AromaticRings": 0.8,
}

def add_property_score(target, reference, method="kde"):
    target = target.copy()
    score_parts = {}
    for col in property_weights:
        if method == "kde":
            score_parts[f"prop_{col}"] = kde_desirability(target[col], reference[col]).values
        else:
            score_parts[f"prop_{col}"] = gaussian_desirability(target[col], reference[col]).values
    part_df = pd.DataFrame(score_parts)
    weights = {f"prop_{k}": v for k, v in property_weights.items()}
    target["property_score"] = weighted_geometric_mean(part_df, weights)
    return pd.concat([target.reset_index(drop=True), part_df], axis=1)

agro = add_property_score(agro, agro, method="kde")
drug = add_property_score(drug, agro, method="kde")
zinc = add_property_score(zinc, agro, method="kde")

score_plot = pd.concat([
    agro[["source", "property_score"]],
    drug[["source", "property_score"]],
    zinc[["source", "property_score"]],
])

plt.figure(figsize=(8, 4))
if sns:
    sns.kdeplot(data=score_plot, x="property_score", hue="source", common_norm=False)
else:
    for group, part in score_plot.groupby("source"):
        plt.hist(part["property_score"], bins=40, alpha=0.35, density=True, label=group)
    plt.legend()
plt.title("Property score distribution")
plt.show()


## 4. Scaffold Score: Murcko Scaffold

Murcko scaffold는 graph algorithm 관점과 연결된다. Side chain을 제거하고 molecular core topology를 비교한다.

Score idea:

- Agrochemical에서 자주 관찰되는 scaffold는 높은 점수
- Drug comparison set에서도 흔한 일반적인 scaffold는 penalty
- 의미 있는 scaffold가 없거나 chain 형태인 경우 낮은 baseline score


In [ ]:
def murcko_smiles(mol):
    try:
        scaf = MurckoScaffold.GetScaffoldForMol(mol)
        if scaf is None or scaf.GetNumAtoms() == 0:
            return ""
        return Chem.MolToSmiles(scaf, isomericSmiles=False)
    except Exception:
        return ""

for df in [agro, drug, zinc]:
    df["scaffold"] = df["mol"].map(murcko_smiles)

agro_scaf = agro["scaffold"].value_counts()
drug_scaf = drug["scaffold"].value_counts()
scaffold_table = pd.DataFrame({
    "agro_count": agro_scaf,
    "drug_count": drug_scaf,
}).fillna(0)
scaffold_table = scaffold_table[scaffold_table.index != ""]
scaffold_table["agro_freq"] = scaffold_table["agro_count"] / max(1, len(agro))
scaffold_table["drug_freq"] = scaffold_table["drug_count"] / max(1, len(drug))
scaffold_table["log_enrichment"] = np.log2((scaffold_table["agro_freq"] + EPS) / (scaffold_table["drug_freq"] + EPS))
scaffold_table["scaffold_score_ref"] = (
    np.log1p(scaffold_table["agro_count"]) / np.log1p(scaffold_table["agro_count"].max())
) * (1 / (1 + np.exp(-scaffold_table["log_enrichment"])))
scaffold_table = scaffold_table.sort_values(["scaffold_score_ref", "agro_count"], ascending=False)

display(scaffold_table.head(20))

plt.figure(figsize=(10, 4))
scaffold_table.head(20)["agro_count"].sort_values().plot(kind="barh")
plt.title("Top agrochemical Murcko scaffolds")
plt.xlabel("Agrochemical count")
plt.show()


In [ ]:
def assign_scaffold_score(df):
    df = df.copy()
    score_map = scaffold_table["scaffold_score_ref"].to_dict()
    df["scaffold_score"] = df["scaffold"].map(score_map).fillna(0.05)
    df.loc[df["scaffold"] == "", "scaffold_score"] = 0.02
    return df

agro = assign_scaffold_score(agro)
drug = assign_scaffold_score(drug)
zinc = assign_scaffold_score(zinc)

plt.figure(figsize=(8, 4))
tmp = pd.concat([agro, drug, zinc], ignore_index=True)
if sns:
    sns.kdeplot(data=tmp, x="scaffold_score", hue="source", common_norm=False)
else:
    for group, part in tmp.groupby("source"):
        plt.hist(part["scaffold_score"], bins=40, alpha=0.35, density=True, label=group)
    plt.legend()
plt.title("Scaffold score distribution")
plt.show()


### 해석: scaffold analysis

- Murcko scaffold는 side chain을 제거한 core graph이므로, 분자의 핵심 골격이 agrochemical reference에서 반복적으로 나타나는지 확인하는 데 적합하다.
- Agrochemical에서 enriched된 scaffold는 `agro_count`가 높고 `drug_count` 대비 `log_enrichment`가 양수인 scaffold로 정의한다.
- Drug comparison set에서도 흔한 scaffold는 생리활성 small molecule 전반에서 흔한 일반 scaffold일 가능성이 있으므로 penalty를 준다. 이 방식은 “그냥 흔한 약물형 scaffold”와 “agrochemical 쪽에 더 특이적인 scaffold”를 구분한다.
- Scaffold score는 EDC-like ranking에서 구조적 설명의 중심 역할을 한다. 어떤 후보가 높은 점수를 받았다면, 그 후보가 agrochemical에서 반복적으로 관찰되는 core structure를 공유한다는 근거를 제시할 수 있다.
- Scaffold가 없는 작은 aliphatic molecule이나 reference에 거의 없는 scaffold는 낮은 baseline score를 부여한다. 이는 지나치게 단순하거나 근거 없는 후보가 top hit으로 올라오는 것을 막기 위한 설계이다.


## 5. Similarity Score: Tanimoto Fingerprint

이 단계는 fingerprint와 Tanimoto similarity 수업 내용과 연결된다.

Morgan, RDKit, MACCS fingerprint를 비교한다. 최종 score에는 하나의 fingerprint를 선택하거나 여러 fingerprint의 ensemble average를 사용할 수 있다.


In [ ]:
def fingerprint(mol, kind="morgan"):
    if kind == "morgan":
        return rdMolDescriptors.GetMorganFingerprintAsBitVect(mol, radius=2, nBits=2048)
    if kind == "rdkit":
        return Chem.RDKFingerprint(mol)
    if kind == "maccs":
        return MACCSkeys.GenMACCSKeys(mol)
    raise ValueError(kind)

def max_tanimoto_to_reference(query_fps, ref_fps):
    scores = []
    for fp in query_fps:
        sims = DataStructs.BulkTanimotoSimilarity(fp, ref_fps)
        scores.append(max(sims) if sims else 0.0)
    return np.array(scores)

def add_similarity_score(target, reference, kind="morgan", ref_limit=2000):
    target = target.copy()
    ref = reference.sample(min(ref_limit, len(reference)), random_state=RANDOM_SEED) if len(reference) > ref_limit else reference
    ref_fps = [fingerprint(m, kind) for m in ref["mol"]]
    target_fps = [fingerprint(m, kind) for m in target["mol"]]
    target[f"sim_{kind}"] = max_tanimoto_to_reference(target_fps, ref_fps)
    return target

for kind in ["morgan", "rdkit", "maccs"]:
    agro = add_similarity_score(agro, agro, kind=kind)
    drug = add_similarity_score(drug, agro, kind=kind)
    zinc = add_similarity_score(zinc, agro, kind=kind)

for df in [agro, drug, zinc]:
    df["similarity_score"] = df[["sim_morgan", "sim_rdkit", "sim_maccs"]].mean(axis=1).clip(EPS, 1.0)

sim_df = pd.concat([
    agro.assign(group="Agrochemical"),
    drug.assign(group="Drug comparison"),
    zinc.assign(group="ZINC candidate"),
], ignore_index=True)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, ["sim_morgan", "sim_rdkit", "sim_maccs"]):
    if sns:
        sns.kdeplot(data=sim_df, x=col, hue="group", common_norm=False, ax=ax)
    else:
        for group, part in sim_df.groupby("group"):
            ax.hist(part[col], bins=40, alpha=0.35, density=True, label=group)
        ax.legend(fontsize=8)
    ax.set_title(col)
plt.tight_layout()
plt.show()


### 해석: fingerprint comparison and similarity distribution

- Morgan fingerprint는 원자 주변의 circular environment를 반영하므로 치환기 주변의 local substructure와 bioisosteric-like pattern을 잘 잡는다. Agrochemical 후보 screening의 기본 similarity로 적합하다.
- RDKit fingerprint는 path 기반 fingerprint이므로 ring path, linker, 전체 연결 패턴 차이를 비교하는 데 유용하다.
- MACCS keys는 predefined structural key 기반이라 해석이 쉽고, 특정 functional group 또는 ring pattern의 존재 여부를 빠르게 비교하기 좋다.
- 최종 similarity score는 `sim_morgan`, `sim_rdkit`, `sim_maccs`의 평균을 사용한다. 하나의 fingerprint만 사용하면 특정 구조 표현 방식에 편향될 수 있으므로, local environment, path pattern, interpretable key를 함께 반영한다.
- Similarity distribution 해석에서 agrochemical은 reference self-similarity 때문에 높은 점수 영역이 많고, drug comparison은 더 낮거나 넓은 분포를 보일 것으로 예상된다. ZINC 후보 중 final score가 높은 분자는 agrochemical과의 structural proximity가 있으면서도 property/scaffold/fragment 근거가 동시에 높은 분자이다.


## 6. Fragment Score: SMARTS Enrichment

SMARTS fragment는 설명 가능한 substructure evidence를 제공한다.

아래 fragment library는 시작점이다. 수업 자료와 화학적 판단을 바탕으로 수정하거나 추가할 수 있다. EDC-like 해석에서는 phenol, halogenated aromatic ring, anilide/urea/carbamate, triazine, nitro aromatic, organophosphate 관련 motif에 주목한다.


In [ ]:
SMARTS_LIBRARY = {
    "phenol": "c[OH]",
    "aniline": "c[NH2]",
    "halogenated_aromatic": "c[F,Cl,Br,I]",
    "nitro_aromatic": "c[N+](=O)[O-]",
    "amide": "C(=O)N",
    "urea": "NC(=O)N",
    "carbamate": "OC(=O)N",
    "ester": "C(=O)O",
    "triazine": "n1cncnc1",
    "phosphate": "P(=O)(O)(O)",
    "sulfone_sulfonamide": "S(=O)(=O)",
    "nitrile": "C#N",
    "pyridine": "n1ccccc1",
    "azole": "[nH,n]1cccc1",
}

SMARTS_MOLS = {name: Chem.MolFromSmarts(sma) for name, sma in SMARTS_LIBRARY.items()}
SMARTS_MOLS = {k: v for k, v in SMARTS_MOLS.items() if v is not None}

def add_fragment_flags(df):
    df = df.copy()
    for name, patt in SMARTS_MOLS.items():
        df[f"frag_{name}"] = df["mol"].map(lambda m: int(m.HasSubstructMatch(patt)))
    return df

agro = add_fragment_flags(agro)
drug = add_fragment_flags(drug)
zinc = add_fragment_flags(zinc)

frag_cols = [c for c in agro.columns if c.startswith("frag_")]

def enrichment_table(pos, neg, frag_cols):
    rows = []
    for col in frag_cols:
        a = int(pos[col].sum())
        b = int(len(pos) - a)
        c = int(neg[col].sum())
        d = int(len(neg) - c)
        odds = ((a + 0.5) / (b + 0.5)) / ((c + 0.5) / (d + 0.5))
        rows.append({
            "fragment": col.replace("frag_", ""),
            "agro_rate": a / max(1, len(pos)),
            "drug_rate": c / max(1, len(neg)),
            "odds_ratio": odds,
            "log2_enrichment": np.log2(odds),
            "agro_count": a,
            "drug_count": c,
        })
    return pd.DataFrame(rows).sort_values("log2_enrichment", ascending=False)

frag_enrich = enrichment_table(agro, drug, frag_cols)
display(frag_enrich)

plt.figure(figsize=(9, 5))
plot_frag = frag_enrich.sort_values("log2_enrichment")
plt.barh(plot_frag["fragment"], plot_frag["log2_enrichment"])
plt.axvline(0, color="black", linewidth=1)
plt.xlabel("log2 enrichment: agrochemical vs drug")
plt.title("SMARTS enrichment")
plt.show()


In [ ]:
fragment_weight_map = {
    "phenol": 1.2,
    "halogenated_aromatic": 1.2,
    "nitro_aromatic": 0.8,
    "amide": 0.6,
    "urea": 1.0,
    "carbamate": 1.0,
    "triazine": 1.3,
    "phosphate": 1.0,
    "sulfone_sulfonamide": 0.7,
    "nitrile": 0.7,
    "pyridine": 0.6,
    "azole": 0.6,
}

enrich_map = frag_enrich.set_index("fragment")["log2_enrichment"].clip(lower=0).to_dict()

def add_fragment_score(df):
    df = df.copy()
    raw = np.zeros(len(df), dtype=float)
    hit_names = []
    for idx, row in df.iterrows():
        hits = []
        total = 0.0
        for col in frag_cols:
            frag = col.replace("frag_", "")
            if row[col] == 1:
                contribution = fragment_weight_map.get(frag, 0.5) * (1.0 + enrich_map.get(frag, 0.0))
                total += contribution
                hits.append(frag)
        raw[idx] = total
        hit_names.append(", ".join(hits))
    df["fragment_hits"] = hit_names
    df["fragment_score"] = (1 - np.exp(-raw / 3.0)).clip(0.02, 1.0)
    return df

agro = add_fragment_score(agro)
drug = add_fragment_score(drug)
zinc = add_fragment_score(zinc)

tmp = pd.concat([agro, drug, zinc], ignore_index=True)
plt.figure(figsize=(8, 4))
if sns:
    sns.kdeplot(data=tmp, x="fragment_score", hue="source", common_norm=False)
else:
    for group, part in tmp.groupby("source"):
        plt.hist(part["fragment_score"], bins=40, alpha=0.35, density=True, label=group)
    plt.legend()
plt.title("Fragment score distribution")
plt.show()


### 해석: SMARTS enrichment

- SMARTS enrichment는 agrochemical positive set에서 특정 fragment가 drug comparison set보다 얼마나 자주 나타나는지 보는 분석이다.
- 본 템플릿에서는 phenol, halogenated aromatic, nitro aromatic, amide, urea, carbamate, triazine, phosphate, sulfone/sulfonamide, nitrile, pyridine, azole 등의 pattern을 시작점으로 사용한다.
- EDC-like 설명에 중요한 fragment는 hormone receptor binding 또는 endocrine-related concern과 연결될 수 있는 phenol-like motif, hydrophobic/halogenated aromatic motif, pesticide class에서 자주 등장하는 urea/carbamate/triazine/phosphate motif 등이다.
- Fragment score는 단순히 fragment가 하나라도 있으면 1점을 주는 방식이 아니라, agrochemical vs drug enrichment가 높은 fragment에 더 큰 기여도를 주도록 설계했다.
- 따라서 top candidate 설명에서는 “어떤 SMARTS hit가 있었는지”와 “그 fragment가 agrochemical reference에서 enriched되었는지”를 함께 제시해야 한다.


---
# Task 3. Score 평가 — Goodness of the score (30점)

**문제**: 본인이 만든 score 함수가 양성과 음성을 얼마나 잘 구분하는가?

Score의 goodness는 단일 숫자만으로 판단하지 않는다. 다음 시각화와 통계량을 함께 사용한다.

- 양성/agrochemical, 비교군/drug, screening/ZINC의 score distribution 비교
- ROC curve와 AUC: agrochemical과 drug comparison을 얼마나 구분하는지 평가
- Precision-Recall curve와 average precision: high-score 후보의 신뢰도 확인
- Subscore별 distribution: property, scaffold, similarity, fragment 중 어떤 근거가 구분에 기여했는지 확인
- Top ZINC 구조 이미지와 explanation: 후보가 왜 선택되었는지 화학적으로 설명

## 📝 Score 평가 해석 및 답안

### 1. score가 좋다면 어떤 결과가 예상되는가?

좋은 score라면 agrochemical positive set의 final_score 분포가 drug comparison set보다 오른쪽으로 이동해야 한다. 즉, agrochemical은 높은 score 영역에 많이 분포하고 drug comparison은 낮거나 중간 score 영역에 더 많이 분포해야 한다.

예상되는 시각화 결과는 다음과 같다.

- **Histogram/KDE:** agrochemical score distribution의 중심이 drug보다 높다.
- **ROC curve:** random classifier의 대각선보다 위에 위치하고 AUC가 0.5보다 충분히 높다.
- **PR curve:** high-score 영역에서 precision이 유지된다.
- **Subscore comparison:** final score 차이가 property 하나 때문이 아니라 scaffold, similarity, fragment evidence의 조합으로 설명된다.

### 2. 어떤 기준으로 goodness를 판단하는가?

본 프로젝트의 score는 black-box classifier가 아니므로 accuracy만 보는 것은 적절하지 않다. Goodness는 다음 세 가지를 함께 만족해야 한다.

1. **Separation:** agrochemical과 drug comparison의 final_score 분포가 어느 정도 분리되는가?
2. **Ranking quality:** top-ranked ZINC 후보들이 agrochemical-like property, scaffold, similarity, fragment evidence를 실제로 갖는가?
3. **Explainability:** 높은 점수를 받은 이유를 descriptor, scaffold, Tanimoto similarity, SMARTS fragment로 설명할 수 있는가?

따라서 ROC-AUC가 높더라도 top 후보의 설명 근거가 약하면 좋은 score라고 보기 어렵다. 반대로 AUC가 완벽하지 않아도 top 후보가 일관된 화학적 근거를 갖고 있다면 screening 목적에는 유용할 수 있다.

### 3. 농약에만 있는 구조와 의약품에도 있는 구조는 어떻게 구분하는가?

Drug comparison set에도 aromatic ring, amide, heterocycle처럼 bioactive molecule에 흔한 구조는 많이 존재한다. 따라서 fragment가 존재한다는 사실만으로 agrochemical-like라고 판단하지 않는다.

구분 방법은 다음과 같다.

- Scaffold는 `agro_freq`와 `drug_freq`를 비교하여 agrochemical enrichment를 계산한다.
- SMARTS fragment는 agrochemical rate와 drug rate를 비교하여 odds ratio 또는 log2 enrichment를 계산한다.
- Similarity는 drug 자체와의 유사도가 아니라 agrochemical reference와의 maximum Tanimoto similarity를 기준으로 한다.
- Final score는 enrichment가 없는 일반적인 drug-like feature보다 agrochemical에 상대적으로 특이적인 feature를 더 높게 반영한다.

### 4. ZINC top candidate는 어떻게 해석하는가?

Top ZINC 후보는 final_score만 보고 선택하지 않고, 다음 항목을 함께 확인한다.

- property_score가 높은가?
- agrochemical-enriched Murcko scaffold를 갖는가?
- Morgan/RDKit/MACCS Tanimoto similarity가 높은가?
- EDC/agrochemical relevance가 있는 SMARTS fragment를 포함하는가?

이 네 가지 근거가 동시에 높으면 “EDC-like agrochemical 후보”로 우선순위를 부여한다. 예를 들어 halogenated aromatic scaffold를 갖고, LogP/TPSA가 agrochemical reference 범위에 있으며, known agrochemical과 fingerprint similarity가 높고, phenol/carbamate/triazine 계열 SMARTS hit가 있다면 높은 ranking을 설명할 수 있다.

### 5. 한계

이 평가는 구조 기반 screening이다. 높은 score는 EDC activity의 실험적 증거가 아니라 후보 우선순위이다. 실제 EDC 여부는 receptor binding, reporter assay, toxicity assay, metabolism, exposure data 등을 추가로 확인해야 한다.


## 7. Final Score: Weighted Geometric Mean

최종 점수는 네 가지 subscore를 weighted geometric mean으로 결합한다.

왜 geometric mean을 사용하는가?

- 하나의 subscore만 극단적으로 높아서 최종 점수를 지배하는 문제를 줄인다.
- property, scaffold, similarity, fragment 근거가 모두 어느 정도 지지될 때 높은 점수가 된다.
- 각 subscore가 독립적인 설명 근거로 남는다.


In [ ]:
FINAL_WEIGHTS = {
    "property_score": 0.30,
    "scaffold_score": 0.20,
    "similarity_score": 0.30,
    "fragment_score": 0.20,
}

def add_final_score(df):
    df = df.copy()
    df["final_score"] = weighted_geometric_mean(df, FINAL_WEIGHTS)
    return df

agro = add_final_score(agro)
drug = add_final_score(drug)
zinc = add_final_score(zinc)

final_df = pd.concat([
    agro.assign(group="Agrochemical"),
    drug.assign(group="Drug comparison"),
    zinc.assign(group="ZINC candidate"),
], ignore_index=True)

plt.figure(figsize=(8, 4))
if sns:
    sns.kdeplot(data=final_df, x="final_score", hue="group", common_norm=False)
else:
    for group, part in final_df.groupby("group"):
        plt.hist(part["final_score"], bins=50, alpha=0.35, density=True, label=group)
    plt.legend()
plt.title("Final explainable score distribution")
plt.show()

display(final_df.groupby("group")[list(FINAL_WEIGHTS) + ["final_score"]].describe().round(3))


In [ ]:
if SKLEARN_OK:
    eval_df = pd.concat([
        agro.assign(y=1),
        drug.assign(y=0),
    ], ignore_index=True)
    y = eval_df["y"].values
    s = eval_df["final_score"].values
    auc = roc_auc_score(y, s)
    ap = average_precision_score(y, s)
    print(f"Agrochemical vs drug comparison ROC-AUC = {auc:.3f}")
    print(f"Average precision = {ap:.3f}")

    fpr, tpr, _ = roc_curve(y, s)
    prec, rec, _ = precision_recall_curve(y, s)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(fpr, tpr)
    axes[0].plot([0, 1], [0, 1], "k--", alpha=0.5)
    axes[0].set_title("ROC curve")
    axes[0].set_xlabel("False positive rate")
    axes[0].set_ylabel("True positive rate")
    axes[1].plot(rec, prec)
    axes[1].set_title("Precision-Recall curve")
    axes[1].set_xlabel("Recall")
    axes[1].set_ylabel("Precision")
    plt.tight_layout()
    plt.show()
else:
    print("sklearn is not available; skip ROC/PR evaluation.")


## 8. Top ZINC Candidates and Explanation

이 부분이 최종 보고서에서 가장 중요하다. Top candidate마다 네 subscore, matched scaffold, SMARTS fragment hit, 그리고 왜 EDC-like 후보인지에 대한 짧은 자연어 설명을 제시한다.


In [ ]:
top_cols = [
    "name", "canonical_smiles", "final_score",
    "property_score", "scaffold_score", "similarity_score", "fragment_score",
    "sim_morgan", "sim_rdkit", "sim_maccs",
    "scaffold", "fragment_hits",
]
top_zinc = zinc.sort_values("final_score", ascending=False).head(20)
display(top_zinc[top_cols].round(3))


In [ ]:
def explain_candidate(row):
    lines = []
    lines.append(f"Candidate: {row.get('name', '')}")
    lines.append(f"SMILES: {row['canonical_smiles']}")
    lines.append(f"Final score: {row['final_score']:.3f}")
    lines.append(
        f"Evidence: property={row['property_score']:.3f}, "
        f"scaffold={row['scaffold_score']:.3f}, "
        f"similarity={row['similarity_score']:.3f}, "
        f"fragment={row['fragment_score']:.3f}"
    )
    if row["property_score"] > 0.6:
        lines.append("- Physicochemical properties fall inside the agrochemical-like reference distribution.")
    else:
        lines.append("- Physicochemical properties only partially match the agrochemical reference distribution.")
    if row["scaffold_score"] > 0.3:
        lines.append(f"- Murcko scaffold appears in agrochemical-enriched scaffolds: {row['scaffold']}")
    else:
        lines.append("- Murcko scaffold is rare or absent in the agrochemical reference set.")
    if row["similarity_score"] > 0.4:
        lines.append("- Fingerprint Tanimoto similarity suggests structural proximity to known agrochemicals.")
    else:
        lines.append("- Fingerprint similarity is modest; other evidence should be checked carefully.")
    if row["fragment_hits"]:
        lines.append(f"- SMARTS evidence: {row['fragment_hits']}")
    else:
        lines.append("- No selected SMARTS alert was detected.")
    lines.append("Conclusion: EDC-like hypothesis is supported when multiple independent evidence blocks agree.")
    return "\n".join(lines)

for _, row in top_zinc.head(5).iterrows():
    print(explain_candidate(row))
    print("-" * 80)


In [ ]:
top_mols = list(top_zinc.head(12)["mol"])
legends = [
    f"{r['name']}\nscore={r['final_score']:.2f}"
    for _, r in top_zinc.head(12).iterrows()
]
img = Draw.MolsToGridImage(top_mols, molsPerRow=4, subImgSize=(260, 180), legends=legends)
img


## 9. Weight Sensitivity Experiment

Weight가 바뀌어도 top candidate가 안정적인지 확인한다. 이 실험은 scoring function이 임의적인 weight 선택에 지나치게 의존하는지 평가하기 위한 것이다.


In [ ]:
weight_sets = {
    "balanced": {"property_score": 0.25, "scaffold_score": 0.25, "similarity_score": 0.25, "fragment_score": 0.25},
    "property_similarity": {"property_score": 0.35, "scaffold_score": 0.15, "similarity_score": 0.35, "fragment_score": 0.15},
    "structure_alert": {"property_score": 0.20, "scaffold_score": 0.30, "similarity_score": 0.20, "fragment_score": 0.30},
    "current": FINAL_WEIGHTS,
}

sensitivity = zinc[["name", "canonical_smiles"] + list(FINAL_WEIGHTS)].copy()
for label, weights in weight_sets.items():
    sensitivity[f"score_{label}"] = weighted_geometric_mean(sensitivity, weights)

display(sensitivity.sort_values("score_current", ascending=False).head(15))

rank_cols = [f"score_{k}" for k in weight_sets]
plt.figure(figsize=(8, 5))
if sns:
    sns.heatmap(sensitivity.sort_values("score_current", ascending=False).head(20)[rank_cols], cmap="viridis")
else:
    plt.imshow(sensitivity.sort_values("score_current", ascending=False).head(20)[rank_cols], aspect="auto")
    plt.colorbar()
plt.title("Weight sensitivity for top ZINC candidates")
plt.show()


## 10. Student Report Cells

### 최종 scoring system 정의

- **Property score:** RDKit descriptor(MW, LogP, TPSA, HBD, HBA, RotB, AromaticRings)에 대해 agrochemical reference distribution을 만들고, Gaussian 또는 KDE desirability를 계산한다. 각 descriptor desirability는 geometric mean으로 결합한다.
- **Scaffold score:** Murcko scaffold를 추출한 뒤 agrochemical frequency와 drug comparison frequency를 비교한다. Agrochemical에서 자주 나타나고 drug에는 덜 나타나는 scaffold일수록 높은 점수를 준다.
- **Similarity score:** Morgan, RDKit, MACCS fingerprint를 각각 계산하고 agrochemical reference set과의 maximum Tanimoto similarity를 구한다. 세 similarity의 평균을 최종 similarity score로 사용한다.
- **Fragment score:** SMARTS library를 이용해 fragment hit를 계산하고, agrochemical vs drug enrichment가 높은 fragment에 더 큰 contribution을 준다.
- **Weighted geometric mean weights:** `property_score=0.30`, `scaffold_score=0.20`, `similarity_score=0.30`, `fragment_score=0.20`을 기본값으로 사용한다.

최종 score는 다음과 같다.

```text
Final score = exp(sum_i w_i * log(score_i + eps) / sum_i w_i)
```

Weighted geometric mean을 사용한 이유는 하나의 근거만 높은 분자를 과대평가하지 않기 위해서다. EDC-like 후보로 선택되려면 property, scaffold, similarity, fragment evidence가 함께 지지되어야 한다.

### 왜 EDC-like인가?

Top candidate는 다음 네 가지 근거가 동시에 높을 때 EDC-like agrochemical 후보로 설명한다.

1. **Property evidence:** MW, LogP, TPSA, HBD/HBA, RotB가 agrochemical reference distribution의 high-density 영역에 위치한다. 이는 분자가 알려진 agrochemical과 비슷한 물성 공간에 있다는 의미이다.
2. **Scaffold evidence:** Murcko scaffold가 agrochemical에서 반복적으로 등장하고 drug comparison set에서는 상대적으로 덜 등장한다. 이는 단순한 drug-like molecule이 아니라 agrochemical-like core를 가질 가능성을 보여준다.
3. **Similarity evidence:** Morgan/RDKit/MACCS Tanimoto similarity가 agrochemical reference와 높다. 이는 전체 구조 또는 부분 구조가 알려진 agrochemical과 가깝다는 근거이다.
4. **Fragment evidence:** phenol, halogenated aromatic, carbamate, urea, triazine, phosphate, nitro aromatic 등 EDC/agrochemical relevance가 있는 SMARTS fragment를 포함한다. Fragment hit는 candidate가 왜 선택되었는지 직접 설명하는 substructure evidence가 된다.

따라서 최종적으로 높은 점수를 받은 ZINC 후보는 “물성이 agrochemical-like이고, core scaffold가 agrochemical-enriched이며, 알려진 agrochemical과 fingerprint similarity가 높고, 설명 가능한 SMARTS fragment를 포함하기 때문에” EDC-like 후보로 해석한다.

### Drug comparison set과의 차이

Drug set은 생리활성이 있는 small molecule이라는 점에서 좋은 비교군이지만, agrochemical과 목적과 구조공간이 다르다. Drug에서도 aromatic ring, amide, heterocycle은 흔할 수 있으므로 fragment 존재만으로는 충분하지 않다. 본 score는 drug에도 흔한 scaffold/fragment는 enrichment 계산에서 penalty를 받고, agrochemical에 더 특이적인 scaffold/fragment와 property distribution을 함께 요구한다.

### 한계

이 score는 EDC activity를 실험적으로 증명하지 않는다. EDC-like 가능성을 구조와 물성 기반으로 우선순위화하는 screening score이다. 실제 endocrine disruption 여부는 receptor binding assay, reporter gene assay, in vivo/in vitro toxicity data, metabolism, exposure level 등을 추가로 확인해야 한다.

### 알고리즘 흐름

```mermaid
flowchart TD
    A[Load agrochemical, drug, ZINC data] --> B[Clean and canonicalize SMILES]
    B --> C[Calculate RDKit descriptors]
    B --> D[Extract Murcko scaffolds]
    B --> E[Generate fingerprints]
    B --> F[Match SMARTS fragments]
    C --> G[Property score]
    D --> H[Scaffold enrichment score]
    E --> I[Tanimoto similarity score]
    F --> J[Fragment enrichment score]
    G --> K[Weighted geometric mean]
    H --> K
    I --> K
    J --> K
    K --> L[Rank ZINC candidates]
    L --> M[Explain why each top hit is EDC-like]
```

### 결론

이 시스템은 black-box classifier가 아니다. 각 후보가 선택된 이유를 property, scaffold, similarity, fragment evidence로 추적할 수 있는 explainable scoring system이다. 최종 ranking은 후보 선별을 위한 우선순위이며, 독성 또는 endocrine activity 확정 판정은 추가 실험과 외부 생물학 데이터가 필요하다.
